**상품 대분류 별로 프로모션별 추가되어야 할 수량을 구한 뒤 발주 판매 데이터와 결합**

- 사용 데이터 : 프로모션, 영수증, 발주판매, 상품마스터

- 각 대분류 별로 LR_CD, LR_NM을 설정한뒤 전체 실행 필요

- 데이터 경로 변경 필요

In [2]:
import pandas as pd
import numpy as np
import os
import re

# 전체 데이터 프로모션 수량 수정

## 대분류 필터링

In [ ]:
promotion_df = pd.read_csv('A-02. 프로모션 매출_한글컬럼.csv')
receipt_df = pd.read_csv('A-05. 영수증_한글컬럼.csv')
goods_master = pd.read_csv('(공통)상품마스터_한글컬럼.csv')

In [ ]:
# 프로모션 데이터에 상품분류코드 추가
goods_master['상품코드'] = goods_master['상품코드'].astype(str).str.lstrip('0') # '073235' -> '73235
promotion_df['상품코드'] = promotion_df['상품코드'].astype(str)
promo_merge = pd.merge(promotion_df, goods_master[['상품코드','상품대분류코드','상품중분류코드','상품소분류코드']], on='상품코드', how = 'left')

promo_merge = promo_merge[['기준일자','점포코드','점포명','상품코드','상품명','MM구분',
                            '행사명','행사코드','판매수량','행사판매금액','정상판매금액','행사시작일자','행사종료일자',
                            '영수증번호', '영수증내상품순번','원가','상품대분류코드','상품중분류코드','상품소분류코드']]
receipt_sp = receipt_df[['기준일자','점포코드','상품코드','상품명','판매시각','거래구분코드',
                        '매출금액','매출수량','POS번호','거래번호','매출원가금액','영수증내상품순번',
                        '상품대분류코드','상품중분류코드','상품소분류코드']]

# 6 - 유음료 / 7 - 냉장 / 8 - 빵 / 11 - 과자
LR_CD = 6
LR_NM = '빵'
promo_merge = promo_merge[promo_merge['상품대분류코드']==LR_CD]
receipt_sp = receipt_sp[receipt_sp['상품대분류코드']==LR_CD]

In [ ]:
## 프로모션 데이터 전처리
date_cols = ['기준일자', '행사시작일자', '행사종료일자']

for col in date_cols:
    promo_merge[col] = pd.to_datetime(
        promo_merge[col].astype(str),
        format='%Y%m%d',
        errors='coerce'
    )

promo_merge.loc[:, '점포코드'] = promo_merge['점포코드'].astype(str).str.strip()
promo_merge.loc[:, '상품코드'] = promo_merge['상품코드'].astype(str).str.strip()

promo_merge.loc[:, '행사코드'] = promo_merge['행사코드'].astype(str)
promo_merge.loc[:, '영수증번호'] = promo_merge['영수증번호'].astype(str)

cols_to_convert = ['판매수량','행사판매금액','정상판매금액','원가']

promo_merge.loc[:, cols_to_convert] = (
    promo_merge[cols_to_convert]
    .replace(',', '', regex=True)
    .astype(float)
)
promo_merge['제조업체'] = promo_merge['상품명'].str.split(')').str[0]
promo_merge['제조업체'] = promo_merge['제조업체'].fillna('UNKNOWN')

## 영수증 데이터 전처리
receipt_sp['기준일자'] = pd.to_datetime(
    receipt_sp['기준일자'].astype(str), 
    format='%Y%m%d',      
    errors='coerce'
)

receipt_sp.loc[:, '상품코드'] = receipt_sp['상품코드'].astype(str).str.strip()
receipt_sp.loc[:, '점포코드'] = receipt_sp['점포코드'].astype(str).str.strip()

receipt_sp.loc[:, '거래번호'] = receipt_sp['거래번호'].astype(str)
receipt_sp.loc[:, '거래구분코드'] = receipt_sp['거래구분코드'].astype(str)

cols_to_convert = ['매출금액','매출수량','매출원가금액']

receipt_sp.loc[:, cols_to_convert] = (
    receipt_sp[cols_to_convert]
    .replace(',', '', regex=True)
    .astype(float)
)
receipt_sp['개당원가금액'] = receipt_sp['매출원가금액'] / receipt_sp['매출수량']

receipt_sp['제조업체'] = receipt_sp['상품명'].str.split(')').str[0]
receipt_sp['제조업체'] = receipt_sp['제조업체'].fillna('UNKNOWN')

In [ ]:
# CSV 저장
promo_merge.to_csv(f'{LR_NM}_프로모션.csv', encoding='utf-8-sig', index=False)
receipt_sp.to_csv(f'{LR_NM}_영수증.csv', encoding='utf-8-sig', index=False)

## 프로모션 별 적용되는 상품
- 영수증 데이터 + 프로모션 데이터 결합

In [ ]:
promo_merge = pd.read_csv(f'{LR_NM}/{LR_NM}_프로모션.csv')
receipt_sp = pd.read_csv(f'{LR_NM}/{LR_NM}_영수증.csv')

In [ ]:
receipt_sp_13 = receipt_sp[receipt_sp['거래구분코드'].isin([1,3])] # 판매데이터만 필터링

receipt_sp_13.loc[:, '점포코드'] = receipt_sp_13['점포코드'].astype(str).str.strip()
receipt_sp_13.loc[:, '상품코드'] = receipt_sp_13['상품코드'].astype(str).str.strip()

In [ ]:
# 행사코드, 행사명, 행사 시작/종료일자 기준으로 데이터를 그룹화
promo_item_list = (
    promo_merge.groupby(['행사코드','행사명','행사시작일자','행사종료일자'])
      .agg(
          점포코드=('점포코드', lambda x: sorted(x.unique().tolist())),
          점포목록=('점포명', lambda x: sorted(x.unique().tolist())),
          상품코드목록=('상품코드', lambda x: sorted(x.unique().tolist())),
          상품명목록=('상품명',   lambda x: sorted(x.unique().tolist()))
      )
      .reset_index()
)

# 행사시작일자, 행사종료일자의 '99991231' 값을 '20251231'로 대체
for col in ['행사시작일자', '행사종료일자']:
    promo_item_list[col] = promo_item_list[col].astype(str)
    promo_item_list.loc[promo_item_list[col] == '99991231', col] = '20251231'

promo_item_list.loc[:, '행사코드'] = promo_item_list['행사코드'].astype(str).str.strip()

In [ ]:
promo_key = (
    promo_item_list
        .explode('점포코드')
        .explode('상품코드목록')
        .rename(columns={'상품코드목록': '상품코드'})
        [['행사코드', '점포코드', '상품코드', '행사시작일자', '행사종료일자']]
        .drop_duplicates()
)

# receipt_sp_13 정규화
receipt_sp_13['점포코드'] = receipt_sp_13['점포코드'].astype(str).str.strip()
receipt_sp_13['상품코드'] = receipt_sp_13['상품코드'].astype(str).str.strip()

# promo_key 정규화
promo_key['점포코드'] = promo_key['점포코드'].astype(str).str.strip()
promo_key['상품코드'] = promo_key['상품코드'].astype(str).str.strip()

# 점포코드 + 상품코드 기준으로 영수증과 행사 키 조인
rec_join_all = receipt_sp_13.merge(
    promo_key,
    on=['점포코드', '상품코드'],
    how='inner'
)

date_cols = ['기준일자', '행사시작일자', '행사종료일자']

for col in date_cols:
    rec_join_all[col] = pd.to_datetime(rec_join_all[col], errors='coerce')

# 영수증 기준일자가 행사 기간 내에 포함되는 경우만 필터링
date_mask = (
    (rec_join_all['기준일자'] >= rec_join_all['행사시작일자']) &
    (rec_join_all['기준일자'] <= rec_join_all['행사종료일자'])
)

rec_join = rec_join_all[date_mask].copy()
print('영수증 join 데이터', rec_join.shape)

## 프로모션 데이터 수정

## 행사판매금액이 0인 경우

In [ ]:
# 1. "+" 프로모션 + 행사판매금액 0 필터
promo_filter = promo_merge[
    (promo_merge['행사명'].str.contains(r'\+', regex=True)) &
    (promo_merge['행사판매금액'] == 0)
].sort_values('기준일자').reset_index(drop=True)

receipt_sp_cut = receipt_sp_13.copy()

# 기본값: 원래 판매수량
promo_filter['edit_판매수량'] = promo_filter['판매수량']
promo_filter['행사코드'] = promo_filter['행사코드'].astype(str)
promo_filter['점포코드'] = promo_filter['점포코드'].astype(str)
promo_filter['영수증번호'] = promo_filter['영수증번호'].astype(str)

# 2. 행사코드별 promo_method (1+1 → 1, 2+1 → 2 ...)
code_method = (
    promo_filter[['행사코드', '행사명']]
    .drop_duplicates(subset=['행사코드'])
    .copy()
)
code_method['promo_method'] = (
    code_method['행사명']
    .str.extract(r'(\d+)\s*\+')[0] # '숫자 +' 패턴에서 숫자 추출
    .astype(float)
)

# 3. (기준일자, 거래번호(=영수증), 행사코드)별 영수증 내 매출수량 합계 = receipt_sum
receipt_grp = (
        rec_join
        .groupby(['기준일자', '점포코드', '거래번호', '행사코드'], as_index=False)['매출수량']
        .sum()
        .rename(columns={'매출수량': 'receipt_sum', '거래번호': '영수증번호'})
    )

receipt_grp['행사코드'] = receipt_grp['행사코드'].astype(str)
code_method['행사코드'] = code_method['행사코드'].astype(str)

# 4. 행사코드별 promo_method 붙이기
receipt_grp = receipt_grp.merge(
    code_method[['행사코드', 'promo_method']],
    on='행사코드',
    how='left'
)

# 5. "1이면 그대로, 2이상이면 //2" 로직 적용
receipt_grp = receipt_grp[
    (~receipt_grp['receipt_sum'].isna()) &
    (receipt_grp['receipt_sum'] > 0) &
    (~receipt_grp['promo_method'].isna()) &
    (receipt_grp['promo_method'] > 0)
].copy()

receipt_grp['receipt_sum'] = receipt_grp['receipt_sum'].astype(int)
receipt_grp['promo_method'] = receipt_grp['promo_method'].astype(int)

receipt_grp['total_qty'] = np.where(
    receipt_grp['promo_method'] == 1,
    receipt_grp['receipt_sum'],          # 1+1 : 유료수량 그대로
    receipt_grp['receipt_sum'] // 2      # 2+1 이상 : 유료수량 // 2
)

receipt_grp['행사코드'] = receipt_grp['행사코드'].astype(str)
receipt_grp['점포코드'] = receipt_grp['점포코드'].astype(str)
receipt_grp['영수증번호'] = receipt_grp['영수증번호'].astype(str)

# 6. total_qty를 promo_filter에 붙이기
promo_filter['기준일자'] = pd.to_datetime(promo_filter['기준일자'], format='%Y-%m-%d')

promo_ext = promo_filter.merge(
    receipt_grp[['기준일자', '점포코드', '영수증번호', '행사코드', 'total_qty']],
    on=['기준일자', '점포코드', '영수증번호', '행사코드'],
    how='left'
)
print(promo_ext.shape)
# 7. 그룹별로 total_qty를 균등 분배해서 edit_판매수량 채우기
def allocate_total_qty_fast(group: pd.DataFrame) -> pd.DataFrame:
    # total_qty 동일 그룹이므로 첫 값만 사용
    total = group['total_qty'].values[0]

    # total_qty 없으면 그대로 반환
    if not total or total <= 0 or np.isnan(total):
        return group

    total = int(total)
    n = len(group)

    if n == 1:
        group = group.copy()
        group['edit_판매수량'] = total
        return group

    # 정렬 (상품코드 기준)
    order = np.argsort(group['상품코드'].values)
    base, remainder = divmod(total, n)

    # 초기 배열 생성 (전체 base로 채움)
    new_vals = np.full(n, base, dtype=int)

    # remainder 개수만큼 앞쪽 index에 +1
    if remainder > 0:
        new_vals[order[:remainder]] += 1

    # DataFrame index 순서에 맞게 매핑
    out = group.copy()
    out.loc[group.index[order], 'edit_판매수량'] = new_vals

    return out

promo_fast = (
    promo_ext
    .groupby(['기준일자', '점포코드', '영수증번호', '행사코드'], group_keys=False)
    .apply(allocate_total_qty_fast)
)
promo_fast.head()

## 판매수량 == 0, 행사판매금액 == 0 인 경우

In [ ]:
# 0. 원본 인덱스를 따로 보존
promo_fast = promo_fast.copy()
promo_fast['_idx'] = promo_fast.index

promo_fast['기준일자'] = pd.to_datetime(promo_fast['기준일자'])
promo_fast['점포코드'] = promo_fast['점포코드'].astype(str)
promo_fast['영수증번호'] = promo_fast['영수증번호'].astype(str)
promo_fast['행사코드'] = promo_fast['행사코드'].astype(str)

# 1. 행사코드 단위로, 해당 영수증에서 유료 판매수량 합계
grouped_qty = rec_join.groupby(
    ['기준일자', '점포코드', '거래번호', '행사코드']
)['매출수량'].sum().reset_index()

grouped_qty.rename(columns={'매출수량': '영수증_합계수량'}, inplace=True)

grouped_qty['기준일자'] = pd.to_datetime(grouped_qty['기준일자'])
grouped_qty['점포코드'] = grouped_qty['점포코드'].astype(str)
grouped_qty['거래번호'] = grouped_qty['거래번호'].astype(str)
grouped_qty['행사코드'] = grouped_qty['행사코드'].astype(str)

# 2. promo_fast + grouped_qty merge → promo_merge2
promo_merge2 = promo_fast.merge(
    grouped_qty,
    left_on=['기준일자', '점포코드', '영수증번호', '행사코드'],
    right_on=['기준일자', '점포코드', '거래번호', '행사코드'],
    how='left'
)

# 3. 공통 대상 필터: 행사판매금액==0, 판매수량==0, "숫자+" 패턴 ----
mask = (
    (promo_merge2['행사판매금액'] == 0) &
    (promo_merge2['판매수량'] == 0) &
    (promo_merge2['행사명'].str.contains(r'\d+\s*\+', regex=True))
)

# 4. sku_count 계산  
#   같은 (기준일자, 영수증, 행사코드, 행사명) 그룹 안에서 몇 개 SKU가 있는지
promo_merge2['sku_count'] = promo_merge2.groupby(
    ['기준일자', '점포코드', '행사명', '영수증번호', '행사코드']
)['영수증_합계수량'].transform('count')

promo_merge2['sku_count'].replace(0, 1, inplace=True)  # 0 방지

# 5. N+1 증정 수량 계산 함수
def calc_gift_total(x, n):
    """
    N+1 프로모션의 총 '유효수량' 또는 증정 계산용 base 값
    x : 영수증 합계수량 (유료 수량)
    n : 'N+1'의 N

    - n == 1 → x
    - n >= 2:
        x < n  → N
        x >= n → x + floor(x / n)
    """
    if n == 1:
        return x
    if x < n:
        return n
    else:
        return x + (x // n)

# 6. mask 범위 subset 생성 
sub = promo_merge2.loc[mask].copy()

# N 추출 (행사명에서 숫자 부분)
sub['N'] = sub['행사명'].str.extract(r'(\d+)\s*\+').astype(int)

# 영수증 합계수량 정리
sub['영수증_합계수량'] = sub['영수증_합계수량'].fillna(0).astype(int)

# 7. 그룹별로 N+1 로직 + 나머지 균등 분배 ------------------------

def allocate_edit_qty(group: pd.DataFrame) -> pd.DataFrame:
    """
    같은 (기준일자, 영수증번호, 행사코드, 행사명) 그룹 안에서
    - N+1 계산을 한 뒤
    - SKU 수만큼 균등 분배 + 나머지는 앞 SKU부터 +1씩 배분
    """
    # 공통 값 (같은 그룹 안에서는 모두 동일하다고 가정)
    x = int(group['영수증_합계수량'].iloc[0])  # 유료 수량 합계
    n = int(group['N'].iloc[0])
    sku_cnt = int(group['sku_count'].iloc[0])
    if sku_cnt <= 0:
        sku_cnt = len(group)

    if x <= 0 or sku_cnt == 0:
        return group  # 유효 수량 없으면 그냥 패스

    # 1) N+1 총량 계산
    gift_total = calc_gift_total(x, n)

    # 2) 1+1 과 2+1 이상 분기
    if n == 1:
        # 1+1 계열: 전체 수량 = gift_total (여기선 x와 동일)
        # → 총 수량을 SKU 개수로 최대한 균등하게 나누기
        total_to_dist = gift_total
    else:
        # 2+1 이상: 증정분 = gift_total - 유료수량
        # → "증정수량"만 SKU 개수로 균등 분배
        total_to_dist = gift_total - x
        if total_to_dist <= 0:
            return group

    # 3) 균등 분배 + 나머지 처리
    base, remainder = divmod(total_to_dist, sku_cnt)

    # 상품코드 기준으로 정렬해서 앞에서부터 remainder개에 +1
    group = group.sort_values('상품코드').copy()
    group['edit_판매수량'] = base

    if remainder > 0:
        # 앞에서부터 remainder개에 +1씩 추가
        extra_idx = group.index[:remainder]
        group.loc[extra_idx, 'edit_판매수량'] = base + 1

    # 안전하게 int 캐스팅
    group['edit_판매수량'] = group['edit_판매수량'].astype(int)

    return group

# 그룹 기준: 영수증 단위 + 행사코드 단위
sub = (
    sub
    .groupby(['기준일자', '점포코드', '영수증번호', '행사코드', '행사명'], group_keys=False)
    .apply(allocate_edit_qty)
)

# 8. 결과를 promo_fast에 반영 
sub = sub.set_index('_idx')  # _idx를 인덱스로 설정

promo_fast.loc[sub.index, 'edit_판매수량'] = sub['edit_판매수량']


# 판매수량 수정 데이터와 발주판매데이터와 합치기

## 판매데이터에서 상품대분류 필터링

In [ ]:
order_sale_df = pd.read_csv('A-04. 발주판매_한글컬럼.csv')

In [ ]:
order_sale_df = order_sale_df[order_sale_df['상품대분류코드']==LR_CD] # 6 - 유음료 / 8 - 빵

sale_df = order_sale_df[['기준일자','점포코드','점포명','상품코드','상품명','상품대분류코드','상품중분류코드','상품소분류코드',
                        '매출매가금액','매출이익','매출수량','매출원가금액','매가금액']]

cols_to_convert = ['매출매가금액','매출이익','매출수량','매출원가금액','매가금액']

sale_df.loc[:, cols_to_convert] = (
    sale_df[cols_to_convert]
    .replace(',', '', regex=True)
    .astype(float)
)

sale_df = sale_df[sale_df['매출수량']>0]

In [ ]:
sale_df.to_csv(f'/home/work/Seminar_Folder/A-team/seven_data/{LR_NM}/{LR_NM}_판매.csv', encoding='utf-8-sig', index=False)

## 합치기

In [ ]:
sale_df = pd.read_csv(f'{LR_NM}/{LR_NM}_판매.csv')

In [ ]:
sale_df['기준일자'] = pd.to_datetime(sale_df['기준일자']).dt.date
sale_df['기준일자'] = pd.to_datetime(sale_df['기준일자'])

sale_df['상품코드'] = sale_df['상품코드'].astype(str).str.strip()
sale_df['점포코드'] = sale_df['점포코드'].astype(str).str.strip()

sale_df.sort_values(['기준일자', '점포코드'], inplace=True)

In [ ]:
# 연도, 월 추출
sale_df['year'] = sale_df['기준일자'].dt.year
sale_df['month'] = sale_df['기준일자'].dt.month

# 연-월 조합 생성 (예: 2023-03)
sale_df['year_month'] = sale_df['year'].astype(str) + '-' + sale_df['month'].astype(str)

# 점포 × 상품 기준으로 고유 연-월 개수 계산
sp = (
    sale_df.groupby(['점포코드', '상품코드'])['year_month']
    .nunique()
    .reset_index()
)

# 정렬 (예: 가장 적게 팔린 상품/점포부터)
sp2 = sp.sort_values('year_month', ascending=False)

# 매달 팔린 상품 필터링
steady_cd = sp2[sp2['year_month']==24][['점포코드','상품코드']]

In [ ]:
# 콤보증정 제외
promo_fast_filter = promo_fast[promo_fast['MM구분']!='콤보증정'].reset_index()
promo_fast_filter['상품코드'] = promo_fast_filter['상품코드'].astype(str).str.strip()
promo_fast_filter['점포코드'] = promo_fast_filter['점포코드'].astype(str).str.strip()

# 행사판매금액만 0(증정상품인데 교차증정이 아닌 경우)
promo_fast_sale_not0 = promo_fast_filter[(promo_fast_filter['판매수량']!=0)&(promo_fast_filter['행사판매금액']==0)][['기준일자','점포코드','상품코드','edit_판매수량']]
promo_fast_sale_not0_group = promo_fast_sale_not0.groupby(['기준일자','점포코드','상품코드'])['edit_판매수량'].sum().reset_index()

# 판매수량 == 0, 행사판매금액 == 0(교차증정)
promo_fast_sale_0 = promo_fast_filter[(promo_fast_filter['판매수량']==0)&(promo_fast_filter['행사판매금액']==0)][['기준일자','점포코드','상품코드','edit_판매수량']]
promo_fast_sale_0_group = promo_fast_sale_0.groupby(['기준일자','점포코드','상품코드'])['edit_판매수량'].sum().reset_index()

In [ ]:
df = pd.concat([promo_fast_sale_not0_group, promo_fast_sale_0_group], axis=0)

# 동일한 (기준일자, 점포코드, 상품코드) 기준으로 edit_판매수량 합산
promo_fast_merge = (
    df.groupby(['기준일자', '점포코드', '상품코드'], as_index=False)['edit_판매수량']
      .sum()
)

promo_fast_merge['기준일자'] = pd.to_datetime(promo_fast_merge['기준일자'])

In [ ]:
merged = sale_df.merge(
    promo_fast_merge,
    on=['기준일자','점포코드','상품코드'],
    how='outer'
)

merged['매출수량'] = merged['매출수량'].fillna(0)
merged['edit_판매수량'] = merged['edit_판매수량'].fillna(0)

merged['판매수량_최종'] = merged['매출수량'] + merged['edit_판매수량']


In [ ]:
merged_sale = merged[merged['판매수량_최종']>0] # 판매수량이 0 # 판매수량이 음수(환불) 
merged_sale = merged_sale[merged_sale.notna()] # 해당일, 해당지점, 해당상품이 증정으로만 있는 케이스(당일 판매데이터에 로그가 없음)

In [ ]:
steady_cd['점포코드'] = steady_cd['점포코드'].astype(str)
steady_cd['상품코드'] = steady_cd['상품코드'].astype(str)

merged_sale['점포코드'] = merged_sale['점포코드'].astype(str)
merged_sale['상품코드'] = merged_sale['상품코드'].astype(str)

filtered = steady_cd.merge(merged_sale, on=['점포코드','상품코드'], how='inner')

# 프로모션 학습용 데이터 생성

## 날짜 x 점포 x 상품 단위 데이터 생성

In [ ]:
# 원본 훼손 방지
df = filtered.copy()

# 1) 키 컬럼 타입 통일
df['기준일자'] = pd.to_datetime(df['기준일자'])
df['점포코드'] = df['점포코드'].astype(str)
df['상품코드'] = df['상품코드'].astype(str)

# 2) 날짜 / (점포,상품) 조합 추출
dates = df['기준일자'].drop_duplicates().sort_values()

pairs = df[['점포코드', '상품코드']].drop_duplicates()   # ★ steady_cd에서 온 조합만 유지

# 3) 날짜 × (점포,상품) 조합으로 full grid 생성
date_df = dates.to_frame(name='기준일자')
date_df['key'] = 1
pairs['key'] = 1

full_grid = (
    date_df.merge(pairs, on='key')
           .drop(columns='key')
)

# 4) 원본 인덱스 세팅
df_idx = df.set_index(['기준일자', '점포코드', '상품코드'])

# 5) full_grid 기준으로 재인덱싱
expanded = df_idx.reindex(
    full_grid.set_index(['기준일자','점포코드','상품코드']).index
).reset_index()

# 6) 점포 / 상품 정보 복구
stores = df[['점포코드', '점포명']].drop_duplicates(subset=['점포코드'])
products = df[['상품코드','상품명',
               '상품대분류코드','상품중분류코드','상품소분류코드']] \
             .drop_duplicates(subset=['상품코드'])

expanded = expanded.merge(stores, on='점포코드', how='left')
expanded = expanded.merge(products, on='상품코드', how='left')

# 7) 매출 관련 값 0 채우기
value_cols = [
    '매출매가금액', '매출이익', '매출수량', '매출원가금액', '매가금액',
    'edit_판매수량', '판매수량_최종'
]

for col in value_cols:
    if col in expanded.columns:
        expanded[col] = expanded[col].fillna(0)

final_df = expanded

print(final_df.shape)
final_df.head()


In [ ]:
# 1) 상품 관련 컬럼 우선 정리 (products에서 온 _y 기준이 정확함)
final_df['상품대분류코드'] = final_df['상품대분류코드_y']
final_df['상품중분류코드'] = final_df['상품중분류코드_y']
final_df['상품소분류코드'] = final_df['상품소분류코드_y']

# 2) 필요 없는 컬럼 모두 삭제
final_df_cut = final_df[[
    '기준일자',
    '점포코드',
    '상품코드',
    '상품대분류코드',
    '상품중분류코드',
    '상품소분류코드',
    '판매수량_최종',
    '매출매가금액'
]]


In [ ]:
final_df_cut.to_csv(f'{LR_NM}/{LR_NM}_판매수량_최종.csv', encoding='utf-8-sig',index=False)